## Install and Imports

In [1]:
!pip install transformers datasets evaluate seqeval -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.7 MB/s eta 0:00:00


In [2]:
import numpy as np
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)
import evaluate

## Load Dataset

In [5]:
from datasets import load_dataset

# Load parquet version (correct)
dataset = load_dataset("conll2003", revision="refs/convert/parquet")

# 🔥 Reduce dataset size (FAST training)
train_data = dataset["train"].shuffle(seed=42).select(range(2000))
val_data = dataset["validation"].select(range(500))
test_data = dataset["test"].select(range(500))

print(train_data)
print(val_data)

conll2003/train/0000.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

0000.parquet:   0%|          | 0.00/312k [00:00<?, ?B/s]

0000.parquet:   0%|          | 0.00/283k [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
    num_rows: 2000
})
Dataset({
    features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
    num_rows: 500
})


## Extract Label Info

In [7]:
# POS
pos_labels = dataset["train"].features["pos_tags"].feature.names
pos_id2label = {i: l for i, l in enumerate(pos_labels)}
pos_label2id = {l: i for i, l in enumerate(pos_labels)}

# Chunk
chunk_labels = dataset["train"].features["chunk_tags"].feature.names
chunk_id2label = {i: l for i, l in enumerate(chunk_labels)}
chunk_label2id = {l: i for i, l in enumerate(chunk_labels)}

## Tokenizer

In [8]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-cased")

config.json:   0%|          | 0.00/465 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

## Tokenization + Label Alignment

In [9]:
def align_labels(example, label_key):
    tokenized = tokenizer(
        example["tokens"],
        truncation=True,
        is_split_into_words=True
    )

    labels = []
    word_ids = tokenized.word_ids()
    prev = None

    for word_id in word_ids:
        if word_id is None:
            labels.append(-100)
        elif word_id != prev:
            labels.append(example[label_key][word_id])
        else:
            labels.append(-100)
        prev = word_id

    tokenized["labels"] = labels
    return tokenized

## 7. Apply Preprocessing

In [10]:
pos_train = train_data.map(lambda x: align_labels(x, "pos_tags"))
pos_val = val_data.map(lambda x: align_labels(x, "pos_tags"))

chunk_train = train_data.map(lambda x: align_labels(x, "chunk_tags"))
chunk_val = val_data.map(lambda x: align_labels(x, "chunk_tags"))

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

## 8. Models (DistilBERT)

In [11]:
pos_model = AutoModelForTokenClassification.from_pretrained(
    "distilbert-base-cased",
    num_labels=len(pos_labels),
    id2label=pos_id2label,
    label2id=pos_label2id
).to(device)

chunk_model = AutoModelForTokenClassification.from_pretrained(
    "distilbert-base-cased",
    num_labels=len(chunk_labels),
    id2label=chunk_id2label,
    label2id=chunk_label2id
).to(device)

model.safetensors:   0%|          | 0.00/263M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForTokenClassification LOAD REPORT from: distilbert-base-cased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForTokenClassification LOAD REPORT from: distilbert-base-cased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## 9. Metrics (Shared)

In [12]:
metric = evaluate.load("seqeval")

def compute_metrics(p, id2label):
    preds, labels = p
    preds = np.argmax(preds, axis=2)

    true_preds, true_labels = [], []

    for pred, lab in zip(preds, labels):
        p_list, l_list = [], []
        for p_, l_ in zip(pred, lab):
            if l_ != -100:
                p_list.append(id2label[p_])
                l_list.append(id2label[l_])
        true_preds.append(p_list)
        true_labels.append(l_list)

    res = metric.compute(predictions=true_preds, references=true_labels)

    return {
        "precision": res["overall_precision"],
        "recall": res["overall_recall"],
        "f1": res["overall_f1"],
    }

## 10. Training Arguments


In [15]:
!pip install -U transformers

In [14]:
from transformers import TrainingArguments
import torch

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=3e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_steps=20,
    report_to="none",
    save_strategy="no",
    eval_strategy="epoch",
    fp16=torch.cuda.is_available()
)

In [15]:
data_collator = DataCollatorForTokenClassification(tokenizer)

## 12. Trainers

In [17]:
pos_trainer = Trainer(
    model=pos_model,
    args=training_args,
    train_dataset=pos_train,
    eval_dataset=pos_val,
    data_collator=data_collator,
    compute_metrics=lambda p: compute_metrics(p, pos_id2label)
)

chunk_trainer = Trainer(
    model=chunk_model,
    args=training_args,
    train_dataset=chunk_train,
    eval_dataset=chunk_val,
    data_collator=data_collator,
    compute_metrics=lambda p: compute_metrics(p, chunk_id2label)
)


## 12. Train Models

In [18]:
print("Training POS...")
pos_trainer.train()

print("Training Chunking...")
chunk_trainer.train()

Training POS...


Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,1.155452,0.704786,0.796551,0.761078,0.778410
2,0.525853,0.464718,0.865801,0.846137,0.855856


/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: NNP seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: : seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: IN seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: NN seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: . seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarni

Training Chunking...


Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,0.505237,0.369722,0.845092,0.812684,0.828571
2,0.309722,0.314217,0.867512,0.832965,0.849887


TrainOutput(global_step=126, training_loss=0.8045430211793809, metrics={'train_runtime': 7.9448, 'train_samples_per_second': 503.475, 'train_steps_per_second': 15.859, 'total_flos': 57068070648960.0, 'train_loss': 0.8045430211793809, 'epoch': 2.0})

## 13. Evaluation

In [19]:
print("POS:", pos_trainer.evaluate())
print("Chunk:", chunk_trainer.evaluate())

POS: {'eval_loss': 0.4647181034088135, 'eval_precision': 0.8658008658008658, 'eval_recall': 0.8461367178802048, 'eval_f1': 0.8558558558558559, 'eval_runtime': 0.7007, 'eval_samples_per_second': 713.595, 'eval_steps_per_second': 22.835, 'epoch': 2.0}


Chunk: {'eval_loss': 0.31421735882759094, 'eval_precision': 0.8675115207373272, 'eval_recall': 0.8329646017699115, 'eval_f1': 0.8498871331828443, 'eval_runtime': 0.4885, 'eval_samples_per_second': 1023.632, 'eval_steps_per_second': 32.756, 'epoch': 2.0}


## 14. Inference (Both Models)

In [20]:
from transformers import pipeline

pos_pipe = pipeline("token-classification", model=pos_model, tokenizer=tokenizer)
chunk_pipe = pipeline("token-classification", model=chunk_model, tokenizer=tokenizer)

text = "John works at Google in California"

print("\nPOS:")
for r in pos_pipe(text):
    print(r["word"], "→", r["entity"])

print("\nChunk:")
for r in chunk_pipe(text):
    print(r["word"], "→", r["entity"])


POS:
John → NNP
works → NNS
at → IN
Google → NNP
in → IN
California → NNP

Chunk:
John → B-NP
works → B-VP
at → B-PP
Google → B-NP
in → B-PP
California → B-NP


## **Observations**
The model achieved ~85% F1 score on both POS tagging and chunking tasks.
POS tagging slightly outperformed chunking due to its simpler word-level nature.
Chunking performance remained competitive despite requiring phrase-level context understanding.
## **Key Insights**
Precision was consistently higher than recall, indicating the model is more cautious in predictions.
DistilBERT proved effective even with a reduced dataset, showing strong generalization capability.
Subword handling using -100 masking was critical in ensuring correct training.
## **Challenges Faced**
Aligning labels with subword tokens
Handling ignored tokens (-100)
Balancing training speed vs performance